# Custom Callbacks

This tutorial shows how to extend `trainax` with custom `Callback` subclasses. We implement three progressively richer examples:

1. **EarlyStopping** — stop training when validation loss stops improving.
2. **CSVLogger** — write per-epoch metrics to a CSV file via `FileHandler`.
3. **GradientNormTracker** — record the L2 norm of per-step gradients.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import optax
import equinox as eqx

from trainax import EQXTrainer, JaxLoader, StepOutput, ValStepOutput, Callback
from trainax._types import EpochOutput
from trainax._file_handler import FileHandler

## Hook lifecycle recap

The trainer calls these hooks in order each epoch:

```
on_epoch_start
  for each batch:
    on_train_step_end
  [if val epoch]:
    on_val_start
    for each val batch:
      on_val_end
on_epoch_end
[after all epochs]:
on_train_end
```

Override only the hooks you need; the base class no-ops all of them.

## 1. EarlyStopping

In [ ]:
class EarlyStopping(Callback):
    """Stop training when the monitored metric has not improved for `patience` epochs."""

    def __init__(self, patience: int = 5, monitor: str = "val_loss", name: str = "EarlyStopping"):
        super().__init__(name)
        self.patience = patience
        self.monitor = monitor
        self._best = np.inf
        self._counter = 0
        self.stopped_epoch: int | None = None

    def on_epoch_end(self, model, pbar, epoch, epoch_output: EpochOutput, file_handler):
        current = (
            epoch_output.val_loss
            if self.monitor == "val_loss"
            else epoch_output.train_loss
        )
        if current is None:
            return  # validation not run this epoch

        if current < self._best:
            self._best = current
            self._counter = 0
        else:
            self._counter += 1

        if self._counter >= self.patience:
            self.stopped_epoch = epoch
            # Signal the trainer to stop by setting n_epochs on the pbar
            # (the trainer checks pbar.total vs current n, so we exhaust it)
            pbar.update(pbar.total - pbar.n)
            print(f"\nEarly stopping triggered at epoch {epoch} (patience={self.patience})")

early_stop = EarlyStopping(patience=3, monitor="val_loss")
print("EarlyStopping callback created")

## 2. CSVLogger (using FileHandler)

`FileHandler` holds open file handles during training. Register file paths via `continuous_files=` in the trainer constructor; retrieve handles by key inside `on_epoch_end`.

In [ ]:
class CSVLogger(Callback):
    """Write epoch, train_loss, val_loss to a CSV file."""

    def __init__(self, file_key: str = "metrics_csv", name: str = "CSVLogger"):
        super().__init__(name)
        self._file_key = file_key
        self._header_written = False

    def on_epoch_end(self, model, pbar, epoch, epoch_output: EpochOutput, file_handler: FileHandler):
        f = file_handler[self._file_key]
        if not self._header_written:
            f.write("epoch,train_loss,val_loss\n")
            self._header_written = True
        val = epoch_output.val_loss if epoch_output.val_loss is not None else ""
        f.write(f"{epoch},{epoch_output.train_loss},{val}\n")
        f.flush()

csv_logger = CSVLogger(file_key="metrics_csv")
print("CSVLogger callback created")

## 3. GradientNormTracker

In [ ]:
class GradientNormTracker(Callback):
    """Record the global L2 norm of gradients at each training step."""

    def __init__(self, name: str = "GradNormTracker"):
        super().__init__(name)
        self.grad_norms: list[float] = []

    def on_train_step_end(self, step_output: StepOutput, **kwargs):
        if step_output.gradients is None:
            return
        leaves = jax.tree_util.tree_leaves(step_output.gradients)
        global_norm = float(np.sqrt(sum(np.sum(np.asarray(g) ** 2) for g in leaves)))
        self.grad_norms.append(global_norm)

grad_tracker = GradientNormTracker()
print("GradientNormTracker callback created")

## 4. Putting it all together

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────────────
np.random.seed(42)
x = np.random.randn(400, 8).astype(np.float32)
y = (x @ np.random.randn(8, 1)).astype(np.float32).squeeze(-1)

trainloader = JaxLoader({"x": x[:300], "y": y[:300]}, batch_size=32)
valloader   = JaxLoader({"x": x[300:], "y": y[300:]}, batch_size=32)

# ── Model & step functions ────────────────────────────────────────────────────
model = eqx.nn.Linear(8, 1, key=jax.random.key(0))

def train_step(model, batch):
    def loss_fn(m):
        yhat = jax.vmap(m)(batch["x"]).squeeze(-1)
        return jnp.mean((yhat - batch["y"]) ** 2), yhat
    (loss, yhat), grads = eqx.filter_value_and_grad(loss_fn, has_aux=True)(model)
    return StepOutput(loss=loss, y=batch["y"], yhat=yhat, gradients=grads)

def val_step(model, batch):
    yhat = jax.vmap(model)(batch["x"]).squeeze(-1)
    loss = jnp.mean((yhat - batch["y"]) ** 2)
    return ValStepOutput(loss=loss, y=batch["y"], yhat=yhat)

# ── Trainer ───────────────────────────────────────────────────────────────────
trainer = EQXTrainer(
    n_epochs=20,
    callbacks=[early_stop, csv_logger, grad_tracker],
    continuous_files={"metrics_csv": "/tmp/metrics.csv"},
    val_every=2,
    use_rich=False,
)

model, _ = trainer.train(
    model, optax.adam(1e-3), train_step, trainloader, val_step, valloader
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(grad_tracker.grad_norms)
axes[0].set_title("Gradient L2 norm (per step)")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Norm")

# Read the CSV that CSVLogger wrote
import csv
with open("/tmp/metrics.csv") as f:
    rows = list(csv.DictReader(f))
epochs_csv = [int(r["epoch"]) for r in rows]
train_losses = [float(r["train_loss"]) for r in rows]
val_losses = [float(r["val_loss"]) for r in rows if r["val_loss"]]
val_epochs = [int(r["epoch"]) for r in rows if r["val_loss"]]

axes[1].plot(epochs_csv, train_losses, label="train")
axes[1].plot(val_epochs, val_losses, "o-", label="val")
axes[1].set_title("Loss curves (from CSV)")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

if early_stop.stopped_epoch is not None:
    print(f"Stopped at epoch {early_stop.stopped_epoch}")